# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeddaniyalg/flyrank-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule in plain words:** flag a page for CTR review if it sits in a good position tier
(top_3, page_1, or striking), has enough impression volume to trust the CTR estimate,
and its CTR falls meaningfully below the average CTR of other pages at its own tier.
Bigger the gap and bigger the volume, higher the score, since a big gap on a
high-traffic page is worth more editor time than the same gap on a rarely-seen page.

**Reason code:** one code per row, `ctr_gap_vs_tier`, since this baseline tests exactly
one hypothesis. A future rule could add `stale_and_visible` or `quick_win_volume`
as separate codes, but this week is one rule, one reason.

**Action label:** `review_title_snippet` for anything that scores above zero,
`no_action` otherwise.

Two signals checked before trusting the rule, one bucket table each:
1. **CTR vs position tier**: the signal directly behind FlyRank's CTR-fix logic
   from the session. Claim: CTR is higher at better position tiers.
2. **Volume (impressions_90d)**: the signal behind the quick-win logic. Claim:
   pages need a minimum impression floor before their CTR is trustworthy enough
   to act on; below that floor, CTR swings wildly on tiny denominators.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

os.chdir("./../../")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

visible = df[df["impressions_90d"] >= 100].copy()

tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep", "no_data"]
signal1 = (visible.groupby("position_tier")["ctr"]
           .agg(["mean", "count"])
           .reindex(tier_order)
           .dropna())
print("Signal 1: CTR by position_tier (impressions_90d >= 100)")
print(signal1)

is_monotonic = signal1["mean"].is_monotonic_decreasing
verdict1 = "CONFIRMED" if is_monotonic else "MIXED"
print(f"Verdict 1: {verdict1}")
print("CTR falls as position tier worsens, top_3 highest, deep lowest, matches the session's CTR-fix assumption" if is_monotonic
      else "CTR does not fall cleanly across tiers, check which tier breaks the pattern before trusting it")

vol_bins = [0, 100, 500, 2000, np.inf]
vol_labels = ["under_100", "100_to_500", "500_to_2000", "over_2000"]
df["volume_bucket"] = pd.cut(df["impressions_90d"], bins=vol_bins, labels=vol_labels)

ctr_std_by_vol = df.groupby("volume_bucket", observed=True)["ctr"].agg(["std", "count"])
print("\nSignal 2: CTR volatility (std) by volume bucket")
print(ctr_std_by_vol)

falls_with_volume = ctr_std_by_vol["std"].is_monotonic_decreasing
verdict2 = "CONFIRMED" if falls_with_volume else "MIXED"
print(f"Verdict 2: {verdict2}")
print("CTR gets steadier as volume rises, confirms a volume floor is needed before trusting a CTR estimate" if falls_with_volume
      else "CTR volatility does not fall cleanly with volume, floor choice needs another look")

Signal 1: CTR by position_tier (impressions_90d >= 100)
                   mean   count
position_tier                  
top_3          0.334128   533.0
page_1         0.354760  8633.0
striking       0.255782  5903.0
page_3_5       0.142359  6058.0
deep           0.055415   879.0
Verdict 1: MIXED
CTR does not fall cleanly across tiers, check which tier breaks the pattern before trusting it

Signal 2: CTR volatility (std) by volume bucket
                    std  count
volume_bucket                 
under_100      6.260319   8006
100_to_500     0.594428   5279
500_to_2000    0.300344   6502
over_2000      0.322084  10213
Verdict 2: MIXED
CTR volatility does not fall cleanly with volume, floor choice needs another look


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
MIN_VOLUME = 100

work = visible.copy()
work["tier_avg_ctr"] = work.groupby("position_tier")["ctr"].transform("mean")
work["ctr_gap"] = work["tier_avg_ctr"] - work["ctr"]

good_tier = work["position_tier"].isin(["top_3", "page_1", "striking"]).astype(int)
enough_volume = (work["impressions_90d"] >= MIN_VOLUME).astype(int)
positive_gap = (work["ctr_gap"] > 0).astype(int)

work["score"] = good_tier * enough_volume * positive_gap * work["ctr_gap"] * work["impressions_90d"]
work["reason_code"] = "ctr_gap_vs_tier"
work["action"] = np.where(work["score"] > 0, "review_title_snippet", "no_action")

queue = work.sort_values("score", ascending=False)[
    ["content_id", "client_id", "position_tier", "impressions_90d", "ctr",
     "tier_avg_ctr", "ctr_gap", "score", "reason_code", "action"]
]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Queue written: {queue.shape[0]} rows")
print(f"Flagged for review: {(queue['action'] == 'review_title_snippet').sum()}")
print(queue.head(10))

Queue written: 22006 rows
Flagged for review: 9943
                 content_id          client_id position_tier  impressions_90d  \
6653   content_5fe46e04994d  client_4e07408562        page_1           517715   
26844  content_8c19996aa890  client_4e07408562         top_3           509252   
3394   content_36ff89c8214e  client_19581e27de        page_1           295097   
7678   content_8451fc6f034d  client_d029fa3a95         top_3           272144   
7445   content_c8e9d6ab9013  client_19581e27de        page_1           208678   
6903   content_c84a0ab98e90  client_f369cb89fc        page_1           223271   
26531  content_cb112fce36be  client_19581e27de        page_1           309910   
22028  content_73c54f78c06a  client_f369cb89fc        page_1           213963   
17812  content_aaef01a50def  client_19581e27de        page_1           517109   
29879  content_1a9e894be2e2  client_19581e27de        page_1           416180   

        ctr  tier_avg_ctr   ctr_gap          score      r

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).reset_index(drop=True)

for i, row in top20.iterrows():
    print(f"{i+1}. {row['action']} | content_id={row['content_id']} | tier={row['position_tier']} "
          f"| impressions={row['impressions_90d']:.0f} | ctr={row['ctr']:.2f} vs tier avg {row['tier_avg_ctr']:.2f}")
    print(f"   why: gap of {row['ctr_gap']:.2f} points below its tier average, "
          f"on {row['impressions_90d']:.0f} impressions, score {row['score']:.1f}")
    print(f"   would be wrong if: the low CTR is normal for this page's query intent "
          f"(e.g. informational, no click-through expected), or a SERP feature is stealing clicks unrelated to the title")
    print()

1. review_title_snippet | content_id=content_5fe46e04994d | tier=page_1 | impressions=517715 | ctr=0.14 vs tier avg 0.35
   why: gap of 0.21 points below its tier average, on 517715 impressions, score 111184.3
   would be wrong if: the low CTR is normal for this page's query intent (e.g. informational, no click-through expected), or a SERP feature is stealing clicks unrelated to the title

2. review_title_snippet | content_id=content_8c19996aa890 | tier=top_3 | impressions=509252 | ctr=0.15 vs tier avg 0.33
   why: gap of 0.18 points below its tier average, on 509252 impressions, score 93767.3
   would be wrong if: the low CTR is normal for this page's query intent (e.g. informational, no click-through expected), or a SERP feature is stealing clicks unrelated to the title

3. review_title_snippet | content_id=content_36ff89c8214e | tier=page_1 | impressions=295097 | ctr=0.05 vs tier avg 0.35
   why: gap of 0.30 points below its tier average, on 295097 impressions, score 89933.7
   woul

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
weak = top20[top20["impressions_90d"] < top20["impressions_90d"].median()]
print("Weakest picks in the top 20, lower relative volume than their peers:")
print(weak[["content_id", "position_tier", "impressions_90d", "ctr_gap", "score"]])
print("\nWhy weak: even above the 100-impression floor, a page near that floor has a shakier "
      "CTR estimate than one with thousands of impressions, so its rank could shuffle on a noisy day.")

feature_cols = ["position_tier", "impressions_90d", "ctr", "tier_avg_ctr"]
label_derived = ["trend_direction", "trend_pct", "is_declining_label"]
leak_present = [c for c in label_derived if c in queue.columns]

print(f"\nFeatures used in score: {feature_cols}")
print(f"Label-derived columns checked for and excluded: {label_derived}")
print(f"Any leaked into the queue: {leak_present if leak_present else 'none'}")
print("No future window used, ctr and impressions_90d are both trailing-90-day snapshots, "
      "not last-30 vs prev-30 splits, so there is no forward-looking column in this baseline.")

Weakest picks in the top 20, lower relative volume than their peers:
              content_id position_tier  impressions_90d   ctr_gap  \
4   content_c8e9d6ab9013        page_1           208678  0.354760   
11  content_a7427266c305        page_1           201111  0.244760   
12  content_453722754fea        page_1           140079  0.344760   
13  content_91652435f57a        page_1           159590  0.294760   
14  content_c1fe78bc4e37        page_1           134055  0.324760   
15  content_97a86caf3a3d        page_1           147670  0.284760   
16  content_4a6607efcb46         top_3           128068  0.324128   
17  content_b115f7c74779        page_1           123469  0.324760   
18  content_0919dd345d80        page_1           119217  0.334760   
19  content_e12868d1f396         top_3           149712  0.264128   

           score  
4   74030.532830  
11  49223.856610  
12  48293.586064  
13  47040.691463  
14  43535.653973  
15  42050.456516  
16  41510.370882  
17  40097.748390  


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.